# PROYECTO FINAL:
OLIST E-COMMERCE BRASIL

## Enfoque metodológico general del proyecto

El proyecto se desarrolló siguiendo una secuencia de análisis aplicada al estudio de un ecommerce real. La lógica de trabajo se estructuró en cuatro fases principales: comprensión y auditoría de los datos, limpieza e integración de las distintas fuentes, análisis exploratorio de negocio y análisis de sentimiento sobre las reseñas textuales.

Este enfoque permitió construir progresivamente una base analítica sólida, evitando realizar interpretaciones o visualizaciones sobre datos no validados previamente. De este modo, cada notebook cumple una función específica dentro del flujo del proyecto y aporta resultados que sirven de base para la fase siguiente.

# Metodología del Notebook 1 - Comprensión y auditoría inicial de los datos

## Objetivo

El objetivo de este primer notebook fue comprender la estructura general del dataset y realizar una auditoría inicial de calidad sobre las distintas tablas disponibles. Antes de llevar a cabo cualquier limpieza, integración o análisis de negocio, resultaba necesario identificar la granularidad de cada tabla, sus claves principales, la presencia de valores faltantes, duplicados y la coherencia de las relaciones entre archivos.

## Procedimiento

En esta fase se cargaron todas las tablas del proyecto y se realizó una revisión sistemática de sus dimensiones, nombres de columnas, tipos de datos y contenido general. Posteriormente, se analizaron los valores nulos y los duplicados exactos, así como la unicidad de variables clave como `order_id`, `customer_id`, `customer_unique_id`, `product_id`, `seller_id` y `review_id`.

Además, se revisó el comportamiento de tablas con posibles relaciones uno-a-muchos, como `order_items`, `order_payments` y `order_reviews`, con el fin de identificar si podían integrarse directamente o si requerían un tratamiento previo. También se comprobó la correspondencia básica entre tablas relacionadas, por ejemplo entre pedidos y clientes o entre pedidos y pagos.

## Justificación metodológica

Esta fase era imprescindible para evitar errores de interpretación en etapas posteriores. En un proyecto con múltiples archivos interrelacionados, no es metodológicamente correcto comenzar por el análisis descriptivo sin haber entendido primero la estructura real de los datos. Por ello, este notebook cumple una función diagnóstica y de validación inicial.

## Resultado esperado

El resultado de este notebook fue la identificación de la tabla central del análisis (`orders`), el reconocimiento de tablas de detalle que requerían agregación previa y la detección de los principales problemas de calidad de datos que debían abordarse en la siguiente fase.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [2]:
PATH = "../data/raw/"

customers = pd.read_csv(PATH + "olist_customers_dataset.csv")
geolocation = pd.read_csv(PATH + "olist_geolocation_dataset.csv")
order_items = pd.read_csv(PATH + "olist_order_items_dataset.csv")
order_payments = pd.read_csv(PATH + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(PATH + "olist_order_reviews_dataset.csv")
orders = pd.read_csv(PATH + "olist_orders_dataset.csv")
products = pd.read_csv(PATH + "olist_products_dataset.csv")
sellers = pd.read_csv(PATH + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(PATH + "product_category_name_translation.csv")

In [3]:
print("customers:", customers.shape)
print("geolocation:", geolocation.shape)
print("order_items:", order_items.shape)
print("order_payments:", order_payments.shape)
print("order_reviews:", order_reviews.shape)
print("orders:", orders.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)
print("category_translation:", category_translation.shape)

customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


In [4]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

In [6]:
customers.columns

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')

In [7]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')

In [8]:
order_reviews.columns

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')

In [9]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


## Creación de todas las tablas en un diccionario:

In [10]:
tables = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

### para ver el tamaño de todas las filas, utilizamos un bucle for de la siguiente manera:

In [11]:
for name, df in tables.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


## para ver las columnas de cada tabla, ejecutamos el siguiente bucle:

In [12]:
for name, df in tables.items():
    print(f"\n{name.upper()}")
    print(df.columns.tolist())


CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

GEOLOCATION
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

ORDER_PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

ORDER_REVIEWS
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

PRODUCTS
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_heigh

## Información general de las tablas:

In [13]:
for name, df in tables.items():
    print(f"\n{'='*60}")
    print(f"INFO DE: {name}")
    print(f"{'='*60}")
    df.info()


INFO DE: customers
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB

INFO DE: geolocation
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolo

## Valores nulos en cada tabla:

In [14]:
for name, df in tables.items():
    print(f"\n{name.upper()}")
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0].sort_values(ascending=False)
    print(nulls if len(nulls) > 0 else "No hay nulos")


CUSTOMERS
No hay nulos

GEOLOCATION
No hay nulos

ORDER_ITEMS
No hay nulos

ORDER_PAYMENTS
No hay nulos

ORDER_REVIEWS
review_comment_title      87656
review_comment_message    58247
dtype: int64

ORDERS
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

PRODUCTS
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

SELLERS
No hay nulos

CATEGORY_TRANSLATION
No hay nulos


## Porcentaje de valores nulos por tabla:

In [15]:
for name, df in tables.items():
    print(f"\n{name.upper()}")
    pct_nulls = (df.isnull().mean() * 100).sort_values(ascending=False)
    pct_nulls = pct_nulls[pct_nulls > 0]
    print(pct_nulls if len(pct_nulls) > 0 else "No hay nulos")


CUSTOMERS
No hay nulos

GEOLOCATION
No hay nulos

ORDER_ITEMS
No hay nulos

ORDER_PAYMENTS
No hay nulos

ORDER_REVIEWS
review_comment_title      88.341530
review_comment_message    58.702532
dtype: float64

ORDERS
order_delivered_customer_date    2.981668
order_delivered_carrier_date     1.793023
order_approved_at                0.160899
dtype: float64

PRODUCTS
product_category_name         1.851234
product_name_lenght           1.851234
product_description_lenght    1.851234
product_photos_qty            1.851234
product_weight_g              0.006070
product_length_cm             0.006070
product_height_cm             0.006070
product_width_cm              0.006070
dtype: float64

SELLERS
No hay nulos

CATEGORY_TRANSLATION
No hay nulos


## Valores duplicados exactos:

In [16]:
for name, df in tables.items():
    duplicated_rows = df.duplicated().sum()
    print(f"{name}: filas duplicadas exactas = {duplicated_rows}")

customers: filas duplicadas exactas = 0
geolocation: filas duplicadas exactas = 261831
order_items: filas duplicadas exactas = 0
order_payments: filas duplicadas exactas = 0
order_reviews: filas duplicadas exactas = 0
orders: filas duplicadas exactas = 0
products: filas duplicadas exactas = 0
sellers: filas duplicadas exactas = 0
category_translation: filas duplicadas exactas = 0


## Claves unicas principales:
pretende clasificar cuantos valores unicos existen en cada tabla

In [17]:
print("orders - order_id únicos:", orders["order_id"].nunique(), "de", len(orders))
print("customers - customer_id únicos:", customers["customer_id"].nunique(), "de", len(customers))
print("customers - customer_unique_id únicos:", customers["customer_unique_id"].nunique(), "de", len(customers))
print("products - product_id únicos:", products["product_id"].nunique(), "de", len(products))
print("sellers - seller_id únicos:", sellers["seller_id"].nunique(), "de", len(sellers))
print("reviews - review_id únicos:", order_reviews["review_id"].nunique(), "de", len(order_reviews))
print("reviews - order_id únicos:", order_reviews["order_id"].nunique(), "de", len(order_reviews))

orders - order_id únicos: 99441 de 99441
customers - customer_id únicos: 99441 de 99441
customers - customer_unique_id únicos: 96096 de 99441
products - product_id únicos: 32951 de 32951
sellers - seller_id únicos: 3095 de 3095
reviews - review_id únicos: 98410 de 99224
reviews - order_id únicos: 98673 de 99224


## Pedidos repetidos en cada tabla:

In [18]:
print("Pedidos repetidos en order_items:", (order_items["order_id"].value_counts() > 1).sum())
print("Pedidos repetidos en order_payments:", (order_payments["order_id"].value_counts() > 1).sum())
print("Pedidos repetidos en order_reviews:", (order_reviews["order_id"].value_counts() > 1).sum())

Pedidos repetidos en order_items: 9803
Pedidos repetidos en order_payments: 2961
Pedidos repetidos en order_reviews: 547


## Ejemplos concretos de repeticion:

In [19]:
order_items["order_id"].value_counts().head(10)

order_id
8272b63d03f5f79c56e9e4120aec44ef    21
1b15974a0141d54e36626dca3fdc731a    20
ab14fdcfbe524636d65ee38360e22ce8    20
9ef13efd6949e4573a18964dd1bbe7f5    15
428a2f660dc84138d969ccd69a0ab6d5    15
9bdc4d4c71aa1de4606060929dee888c    14
73c8ab38f07dc94389065f7eba4f297a    14
37ee401157a3a0b28c9c6d0ed8c3b24b    13
2c2a19b5703863c908512d135aa6accc    12
c05d6a79e55da72ca780ce90364abed9    12
Name: count, dtype: int64

In [20]:
order_payments["order_id"].value_counts().head(10)

order_id
fa65dad1b0e818e3ccc5cb0e39231352    29
ccf804e764ed5650cd8759557269dc13    26
285c2e15bebd4ac83635ccc563dc71f4    22
895ab968e7bb0d5659d16cd74cd1650c    21
fedcd9f7ccdc8cba3a18defedd1a5547    19
ee9ca989fc93ba09a6eddc250ce01742    19
4bfcba9e084f46c8e3cb49b0fa6e6159    15
21577126c19bf11a0b91592e5844ba78    15
3c58bffb70dcf45f12bdf66a3c215905    14
4689b1816de42507a7d63a4617383c59    14
Name: count, dtype: int64

In [21]:
order_reviews["order_id"].value_counts().head(10)

order_id
c88b1d1b157a9999ce368f218a407141    3
8e17072ec97ce29f0e1f111e598b0c85    3
df56136b8031ecd28e200bb18e6ddb2e    3
03c939fd7fd3b38f8485a0f95798f1f6    3
5cb890a68b91b6158d69257e4e2bc359    2
2143393cca994a4b8235bc1d67ded772    2
25320e12b3d6e8f54f17389037588bba    2
0176a6846bcb3b0d3aa3116a9a768597    2
d9e44c3fd2ce16086619f299e92e12d8    2
21cb4961399b846b168d0358100f473b    2
Name: count, dtype: int64

## Comprobar si existen relaciones entre tablas:

In [22]:
print("Pedidos en orders que están en order_items:", orders["order_id"].isin(order_items["order_id"]).mean() * 100)
print("Pedidos en orders que están en order_payments:", orders["order_id"].isin(order_payments["order_id"]).mean() * 100)
print("Pedidos en orders que están en order_reviews:", orders["order_id"].isin(order_reviews["order_id"]).mean() * 100)
print("Clientes en orders que están en customers:", orders["customer_id"].isin(customers["customer_id"]).mean() * 100)

Pedidos en orders que están en order_items: 99.22064339658692
Pedidos en orders que están en order_payments: 99.99899437857624
Pedidos en orders que están en order_reviews: 99.22768274655324
Clientes en orders que están en customers: 100.0


## Conclusiones del Data Understanding

- `orders` será nuestra tabla central del análisis.
- `customer_unique_id` se utilizará para estudiar clientes recurrentes.
- `order_items`, `order_payments` y `order_reviews` requieren agregación previa antes del merge a nivel pedido.
- `geolocation` contiene muchos duplicados y deberá procesarse antes de unirse con clientes o sellers.
- Existen valores faltantes relevantes en `orders`, `order_reviews` y `products`.
- En el siguiente notebook se realizará la limpieza, estandarización de tipos y construcción de la tabla analítica final.